<a href="https://colab.research.google.com/github/syoon7079/tjwn/blob/main/11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 0) 설정
!pip -q install pyreadstat
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 200)

DATA_DIR = '/content/drive/MyDrive/hdac_data'            # 팀 공통 경로
SAV_PATH = f'{DATA_DIR}/HN24_ALL.sav'
OUT_DIR  = f'{DATA_DIR}/output'                           # 집계표(md)만 저장
os.makedirs(OUT_DIR, exist_ok=True)

AGE_MIN  = 19                  # 연구대상: 19세 이상 성인
WEIGHT   = 'wt_tot'            # 건강설문 AND 검진 AND 영양 결합 가중치 (이용지침서 표 6-2 p.28, 변수 설명 p.94)
KEEP_NA8 = True                # BD1_11, BS3_1 의 8(비해당)을 별도 범주로 유지 (False면 결측 처리)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 0-1) 추가 변수 후보 찾기 (원자료에서 이름·설명만 확인; 자료 값은 읽지 않음)
# 1인 가구 / 근로 여부 / 신체활동 변수의 실제 이름은 이용지침서·코드북으로 확정해야 합니다.
import pyreadstat
_, _m = pyreadstat.read_sav(SAV_PATH, metadataonly=True)
KEYWORDS = ['가구원', '경제활동', '근로', '직업', '신체활동', '유산소', '운동']
for kw in KEYWORDS:
    hits = {n: l for n, l in _m.column_names_to_labels.items() if l and kw in l}
    print(f'\n[{kw}] 후보 {len(hits)}개')
    for n, l in list(hits.items())[:12]:
        print(f'  {n}: {l}')


[가구원] 후보 1개
  cfam: 가구원수

[경제활동] 후보 2개
  occp: 직업재분류 및 실업/비경제활동인구상태 코드
  EC1_1: 경제활동 상태

[근로] 후보 4개
  EC_stt_2: 종사상지위: 임금근로자 상세
  EC_wht_23: 주당 평균 근로시간
  EC_wht_5: 주/변형근로시간
  EC_wht_6: 변형근로시간:기타상세

[직업] 후보 3개
  occp: 직업재분류 및 실업/비경제활동인구상태 코드
  EC_occp: 표준직업분류 대분류 코드
  EC_lgw_2: 최장일자리: 직업분류코드+실업/비경활상태

[신체활동] 후보 22개
  BE3_71: 고강도 신체활동 여부: 일
  BE3_72: 고강도 신체활동 일수: 일
  BE3_73: 고강도 신체활동 시간(시간): 일
  BE3_74: 고강도 신체활동 시간(분): 일
  BE3_81: 중강도 신체활동 여부: 일
  BE3_82: 중강도 신체활동 일수: 일
  BE3_83: 중강도 신체활동 시간(시간): 일
  BE3_84: 중강도 신체활동 시간(분): 일
  BE3_91: 신체활동 여부: 장소이동
  BE3_92: 신체활동 일수: 장소이동
  BE3_93: 신체활동 시간(시간): 장소이동
  BE3_94: 신체활동 시간(분): 장소이동

[유산소] 후보 1개
  pa_aerobic: 유산소 신체활동 실천율

[운동] 후보 2개
  LQ_1EQL: EuroQoL: 운동능력
  BE5_1: 1주일간 근력운동 일수


In [3]:
# 0-2) 11주차 추가 변수 이름 (위 후보 목록과 코드북으로 확인한 뒤 필요하면 고치기)
EXTRA_VARS = {
    'hh':   'cfam',        # 가구원수 -> 1인 가구 여부 만들기
    'work': 'EC1_1',       # 현재 경제활동(근로) 여부
    'pa':   'pa_aerobic',  # 유산소 신체활동 실천 여부
}

In [4]:
# 1) 변수 정의표의 변수만 불러오기
!pip -q install pyreadstat
import pyreadstat
NEEDED = ['year', 'l_out_fq', 'he_bmi', 'bd1_11', 'region', 'sex', 'age', 'educ', 'incm', 'bs3_1',
          'kstrata', 'psu', WEIGHT] + [v.lower() for v in EXTRA_VARS.values()]   # 11주차: 추가 변수
_, m0 = pyreadstat.read_sav(SAV_PATH, metadataonly=True)
low2real = {c.lower(): c for c in m0.column_names}
absent = [c for c in NEEDED if c not in low2real]
assert not absent, f'원자료에 없는 변수: {absent}'
df, meta = pyreadstat.read_sav(SAV_PATH, usecols=[low2real[c] for c in NEEDED])
df.columns = df.columns.str.lower()
print(f'불러온 자료: {len(df):,}행 x {df.shape[1]}열 | 조사연도:', sorted(df['year'].dropna().unique()))

불러온 자료: 6,997행 x 16열 | 조사연도: [np.float64(2024.0)]


In [5]:
# ==============================================================================
# [셀 1] 10주차 전처리 및 데이터 로드
# ==============================================================================
from google.colab import drive
import os
import numpy as np
import pandas as pd
import pyreadstat

drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data'
SAV_PATH = f'{DATA_DIR}/HN24_ALL.sav'

AGE_MIN = 19
WEIGHT = 'wt_tot'

NEEDED = ['year', 'l_out_fq', 'he_bmi', 'bd1_11', 'region', 'sex', 'age', 'educ', 'incm', 'bs3_1',
          'kstrata', 'psu', WEIGHT]

_, m0 = pyreadstat.read_sav(SAV_PATH, metadataonly=True)
low2real = {c.lower(): c for c in m0.column_names}
df, meta = pyreadstat.read_sav(SAV_PATH, usecols=[low2real[c] for c in NEEDED])
df.columns = df.columns.str.lower()

def num(s): return pd.to_numeric(s, errors='coerce')

# 노출: 외식 빈도 3범주
EXP_VALID = [1, 2, 3, 4, 5, 6, 7]
GRP_MAP   = {5: 'low', 6: 'low', 7: 'low', 4: 'mid', 1: 'high', 2: 'high', 3: 'high'}
df['exp_raw']   = num(df['l_out_fq'])
df['exp_valid'] = df['exp_raw'].isin(EXP_VALID)
df['grp']       = df['exp_raw'].where(df['exp_valid']).map(GRP_MAP)
df['grp']       = pd.Categorical(df['grp'], categories=['low', 'mid', 'high'])

# 결과, 연령, 가중치
df['bmi']   = num(df['he_bmi'])
df['age_n'] = num(df['age'])
df['w']     = num(df[WEIGHT])

# 공변량
df['c_sex']   = num(df['sex']).map({1: '남성', 2: '여성'})
df['c_educ']  = num(df['educ']).mask(num(df['educ']).isin([88, 99])).map({1: '초졸 이하', 2: '초졸 이하', 3: '초졸 이하', 4: '중졸', 5: '고졸', 6: '대졸 이상', 7: '대졸 이상', 8: '대졸 이상'})
df['c_incm']  = num(df['incm']).mask(num(df['incm']).isin([8, 9, 99])).map({1: '하', 2: '중하', 3: '중상', 4: '상'})
df['c_bd1_11']= num(df['bd1_11']).mask(num(df['bd1_11']).isin([9, 99])).map({1: '전혀 마시지 않음', 2: '월 1회 미만', 3: '월 1회 정도', 4: '월 2-4회', 5: '주 2-3회', 6: '주 4회 이상', 8: '비해당(코드 8)'})
df['c_bs3_1'] = num(df['bs3_1']).mask(num(df['bs3_1']).isin([9, 99])).map({1: '매일 흡연', 2: '가끔 흡연', 3: '과거 흡연(현재 비흡연)', 8: '비해당(코드 8)'})

# 설계변수 및 analytic
psu_str = df['psu'].astype(str).str.strip()
df['psu_ok']    = df['psu'].notna() & ~psu_str.isin(['', 'nan', 'None'])
df['design_ok'] = df['w'].notna() & (df['w'] > 0) & df['kstrata'].notna() & df['psu_ok']

df['s1'] = True
df['s2'] = df['age_n'] >= AGE_MIN
df['s3'] = df['s2'] & df['exp_valid']
df['s4'] = df['s3'] & df['bmi'].notna()
df['s5'] = df['s4'] & df['design_ok']
df['analytic'] = df['s5']

print(f"전처리 완료! 최종 분석 대상(analytic=True): {int(df['analytic'].sum()):,}명")
print(df.loc[df['analytic'], 'grp'].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
전처리 완료! 최종 분석 대상(analytic=True): 5,765명
grp
low     3062
high    1906
mid      797
Name: count, dtype: int64


#C-1. 화살표 및 인과 근거 표 출력 코드

In [6]:
# ==============================================================================
# C-1. 화살표 및 인과 근거 표 출력 코드
# ==============================================================================
import pandas as pd
from IPython.display import display, HTML

data = [
    ("연령 → 교육, 소득, 근로, 1인 가구", "출생 코호트에 따라 학력이 다름(시간 순서: 연령·코호트가 먼저). 생애주기에 따라 근로·소득·거주 형태가 달라짐"),
    ("성별 → 소득, 근로", "성별에 따라 경제활동 참여와 소득 수준이 다름"),
    ("교육 → 소득, 근로", "학력이 취업과 소득에 선행"),
    ("근로 → 소득", "근로 여부와 형태가 소득을 결정"),
    ("지역 → 외식 빈도, BMI, 소득, 근로", "도시·농촌에 따른 외식 환경(식당 접근성)과 일자리 구조 차이"),
    ("연령, 성별 → 외식 빈도", "Table 1에서 저빈도군은 평균 연령이 높고(57세 대 43세), 고빈도군은 남성 비율이 높음(66%)"),
    ("교육, 소득 → 외식 빈도", "Table 1에서 고빈도군은 대졸 이상(71%)과 소득 상위가 많음 (자료 확인)"),
    ("근로 → 외식 빈도", "직장인은 점심·회식 등으로 외식 기회가 많음"),
    ("1인 가구 → 외식 빈도, BMI", "혼자 살면 조리 대신 외식·간편식을 택하기 쉽고, 식사 패턴이 BMI에 영향"),
    ("연령, 성별 → BMI", "연령·성별에 따른 체성분과 BMI 분포 차이는 선행연구에서 일관됨"),
    ("교육, 소득 → BMI", "사회경제적 수준과 비만의 관련은 선행연구에서 보고됨"),
    ("근로 → BMI", "근로 형태(좌식, 교대근무)가 체중에 영향"),
    ("연령, 성별, 교육 → 신체활동, 흡연, 음주", "생활습관(운동, 흡연, 음주)은 연령·성별·학력에 따라 분포가 다름 (선행연구)"),
    ("근로 → 음주", "직장 회식 등 근로 여부에 따라 음주 기회가 달라짐"),
    ("U → 외식 빈도, 신체활동, 흡연", "건강 지향 성향은 식생활·운동·흡연에 공통으로 작용한다고 가정 (미측정)"),
    ("신체활동, 흡연 → BMI", "에너지 소비와 체중 조절에 대한 영향"),
    ("외식 ← 음주 → BMI", "평소 음주 빈도나 음주량이 많은 사람일수록 술자리를 겸한 외식을 자주 하게 되며, 동시에 알코올 자체의 열량 섭취로 인해 BMI에도 영향을 미침 (외식과 BMI 양쪽에 영향을 주는 교란요인)"),
    ("외식 → 에너지 섭취 → BMI", "연구 가설의 핵심 경로 (매개)"),
    ("외식 → BMI (직접)", "에너지 섭취 외의 경로(식사 구성·영양소 질, 식사 속도 등)를 통한 영향"),
]

df_c1 = pd.DataFrame(data, columns=["화살표", "근거"])

print("=== [C-1] 화살표 및 인과 근거 표 ===")
html_c1 = df_c1.to_html(index=False, escape=False)
display(HTML(html_c1))

=== [C-1] 화살표 및 인과 근거 표 ===


화살표,근거
"연령 → 교육, 소득, 근로, 1인 가구",출생 코호트에 따라 학력이 다름(시간 순서: 연령·코호트가 먼저). 생애주기에 따라 근로·소득·거주 형태가 달라짐
"성별 → 소득, 근로",성별에 따라 경제활동 참여와 소득 수준이 다름
"교육 → 소득, 근로",학력이 취업과 소득에 선행
근로 → 소득,근로 여부와 형태가 소득을 결정
"지역 → 외식 빈도, BMI, 소득, 근로",도시·농촌에 따른 외식 환경(식당 접근성)과 일자리 구조 차이
"연령, 성별 → 외식 빈도","Table 1에서 저빈도군은 평균 연령이 높고(57세 대 43세), 고빈도군은 남성 비율이 높음(66%)"
"교육, 소득 → 외식 빈도",Table 1에서 고빈도군은 대졸 이상(71%)과 소득 상위가 많음 (자료 확인)
근로 → 외식 빈도,직장인은 점심·회식 등으로 외식 기회가 많음
"1인 가구 → 외식 빈도, BMI","혼자 살면 조리 대신 외식·간편식을 택하기 쉽고, 식사 패턴이 BMI에 영향"
"연령, 성별 → BMI",연령·성별에 따른 체성분과 BMI 분포 차이는 선행연구에서 일관됨


#C-1. 공변량별 역할 및 보정 여부 표 출력 코드

In [7]:
# ==============================================================================
# C-1. 공변량별 역할 및 보정 여부 표 출력 코드
# ==============================================================================
import pandas as pd
from IPython.display import display, HTML

data_role = [
    ("<b>외식 빈도</b> <br>(<code>L_OUT_FQ</code>)", "<b>노출</b>",
     "• 연구의 독립변수(원인 후보)<br>• 주간 섭취 빈도에 따라 저·중·고빈도로 분류됨",
     "- (노출)"),
    ("<b>체질량지수 BMI</b> <br>(<code>HE_BMI</code>)", "<b>결과</b>",
     "• 연구의 종속변수(결과)<br>• 외식으로 인한 열량 섭취 증가의 최종 건강 결과 지표",
     "- (결과)"),
    ("<b>성별</b> (<code>sex</code>)", "<b>교란</b>",
     "<b>근거</b>: 생물학적 특성은 외식 빈도와 대사율·체지방 분포(BMI)에 선행하여 영향을 미침",
     "<b>보정함</b>"),
    ("<b>연령</b> (<code>age</code>)", "<b>교란</b>",
     "<b>근거</b>: 연령은 교육·소득·근로 등에 선행하며 외식 빈도와 BMI(체성분)에 모두 영향을 미침",
     "<b>보정함</b>"),
    ("<b>거주 지역</b> (<code>region</code>)", "<b>교란</b>",
     "<b>근거</b>: 지역별 외식 인프라(식당 접근성)와 거주민의 식생활 환경 차이가 외식 빈도와 BMI에 영향을 미침",
     "<b>보정함</b>"),
    ("<b>교육수준</b> (<code>educ</code>)", "<b>교란</b>",
     "<b>근거</b>: 사회경제적 지위(SES)는 식습관 및 건강 인식을 결정하며 외식 빈도와 비만 유병률에 영향을 미침",
     "<b>보정함</b>"),
    ("<b>소득 4분위</b> (<code>incm</code>)", "<b>교란</b>",
     "<b>근거</b>: 경제적 자원(소득 수준)은 외식 소비 여력과 식품 선택에 직접 작용하며 BMI와도 연관됨",
     "<b>보정함</b>"),
    ("<b>음주 빈도</b> (<code>BD1_11</code>)", "<b>교란</b>",
     "<b>근거</b>: 외식 상황(회식, 모임 등)과 밀접하게 연동되어 외식 빈도에 영향을 미치며, 알코올 열량을 통해 BMI 대사에 관여함",
     "<b>보정함</b>"),
    ("<b>현재 흡연</b> (<code>BS3_1</code>)", "<b>교란</b>",
     "<b>근거</b>: 개인의 건강 관련 라이프스타일 지표로서 식생활 패턴(외식) 및 체중 조절 기전에 공통으로 작용함",
     "<b>보정함</b>"),
    ("<b>1인 가구</b> (<code>cfam</code>)", "<b>교란</b>",
     "식사 방식과 BMI에 모두 영향",
     "<b>모형 2 미포함 (한계)</b>"),
    ("<b>근로 여부</b> (<code>EC1_1</code>)", "<b>교란</b>",
     "외식 기회와 BMI에 모두 영향",
     "<b>모형 2 미포함 (한계)</b>"),
    ("<b>신체활동</b> (<code>pa_aerobic</code>)", "<b>교란 가능 (모호)</b>",
     "U(건강 지향 성향) → 신체활동 → BMI 경로 위 변수이나, U가 미측정이라 교란 여부가 모호함",
     "<b>주 분석 미보정 (민감도 S3에서 추가)</b>"),
    ("<b>에너지 섭취</b>", "<b>매개</b>",
     "연구 가설의 핵심 경로 (외식 → 에너지 섭취 → BMI)",
     "<b>보정 안 함 (총효과 추정 시 과다보정 방지)</b>"),
]

df_role = pd.DataFrame(data_role, columns=["변수명 (원변수)", "DAG 상의 역할", "근거", "총효과 모형 보정 여부"])

print("=== [C-1] 공변량별 역할 및 보정 여부 표 ===")
html_role = df_role.to_html(index=False, escape=False)
display(HTML(html_role))

=== [C-1] 공변량별 역할 및 보정 여부 표 ===


변수명 (원변수),DAG 상의 역할,근거,총효과 모형 보정 여부
외식 빈도 (L_OUT_FQ),노출,• 연구의 독립변수(원인 후보)• 주간 섭취 빈도에 따라 저·중·고빈도로 분류됨,- (노출)
체질량지수 BMI (HE_BMI),결과,• 연구의 종속변수(결과)• 외식으로 인한 열량 섭취 증가의 최종 건강 결과 지표,- (결과)
성별 (sex),교란,근거: 생물학적 특성은 외식 빈도와 대사율·체지방 분포(BMI)에 선행하여 영향을 미침,보정함
연령 (age),교란,근거: 연령은 교육·소득·근로 등에 선행하며 외식 빈도와 BMI(체성분)에 모두 영향을 미침,보정함
거주 지역 (region),교란,근거: 지역별 외식 인프라(식당 접근성)와 거주민의 식생활 환경 차이가 외식 빈도와 BMI에 영향을 미침,보정함
교육수준 (educ),교란,근거: 사회경제적 지위(SES)는 식습관 및 건강 인식을 결정하며 외식 빈도와 비만 유병률에 영향을 미침,보정함
소득 4분위 (incm),교란,근거: 경제적 자원(소득 수준)은 외식 소비 여력과 식품 선택에 직접 작용하며 BMI와도 연관됨,보정함
음주 빈도 (BD1_11),교란,"근거: 외식 상황(회식, 모임 등)과 밀접하게 연동되어 외식 빈도에 영향을 미치며, 알코올 열량을 통해 BMI 대사에 관여함",보정함
현재 흡연 (BS3_1),교란,근거: 개인의 건강 관련 라이프스타일 지표로서 식생활 패턴(외식) 및 체중 조절 기전에 공통으로 작용함,보정함
1인 가구 (cfam),교란,식사 방식과 BMI에 모두 영향,모형 2 미포함 (한계)


#완전 사례 표시

In [8]:
# ==============================================================================
# [셀 2] 완전 사례(Complete Case) 분석 대상 인원 확인 및 Table 1 비교
# ==============================================================================
import pandas as pd

# 보정변수: 원변수가 아니라 정제한 c_* 변수 사용 (88·99·9 등 무응답은 이미 NaN 처리됨)
COVS = ['c_sex', 'region', 'c_educ', 'c_incm', 'c_bd1_11', 'c_bs3_1', 'age_n']

# 1) 변수별 결측 수 확인 (Table 1의 결측 수와 같아야 함)
print("=== 분석 대상(analytic) 내 변수별 결측 수 ===")
print(df.loc[df['analytic'], COVS].isna().sum())

# 2) 완전 사례 표시
df['complete_case'] = df['analytic'] & df[COVS].notna().all(axis=1)

n_table1   = int(df['analytic'].sum())
n_complete = int(df['complete_case'].sum())
diff_count = n_table1 - n_complete

print("\n=== [완전 사례 분석 대상 인원 확인] ===")
print(f"1) Table 1 분석 대상 인원 (analytic=True): {n_table1:,} 명")
print(f"2) 보정변수 완전 사례 인원 (Complete Case): {n_complete:,} 명")
print(f"3) 인원 차이: {diff_count:,} 명 (Table 1 대비 공변량 결측으로 제외된 수)")
print("========================================")

=== 분석 대상(analytic) 내 변수별 결측 수 ===
c_sex         0
region        0
c_educ      148
c_incm       28
c_bd1_11    140
c_bs3_1     141
age_n         0
dtype: int64

=== [완전 사례 분석 대상 인원 확인] ===
1) Table 1 분석 대상 인원 (analytic=True): 5,765 명
2) 보정변수 완전 사례 인원 (Complete Case): 5,602 명
3) 인원 차이: 163 명 (Table 1 대비 공변량 결측으로 제외된 수)


#C-2 복합표본 가중 선형회귀분석 (모형 1 & 모형 2) 및 Table 2 출력 코드

In [9]:
# ==============================================================================
# [C-2 수정] 모형 1 및 모형 2 (기준집단: 저빈도 'low') + 검증 + Table 2
# ==============================================================================
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

COVS = ['c_sex', 'region', 'c_educ', 'c_incm', 'c_bd1_11', 'c_bs3_1', 'age_n']

# 1) 완전 사례 서브셋
valid_mask = (df['analytic'] & df['grp'].notna() & df['bmi'].notna()
              & df['design_ok'] & df[COVS].notna().all(axis=1))
sub_df = df[valid_mask].copy()
sub_df['grp'] = pd.Categorical(sub_df['grp'], categories=['low', 'mid', 'high'])
n_table1, n_complete = int(df['analytic'].sum()), len(sub_df)
diff_count = n_table1 - n_complete

# 2) 모형 적합 (WLS + psu 군집 강건 SE)
def fit_wls(formula, data):
    return smf.wls(formula, data=data, weights=data['w']).fit(
        cov_type='cluster', cov_kwds={'groups': data['psu']})

EXPO = "C(grp, Treatment(reference='low'))"
COV_TERMS = "C(c_sex) + C(region) + C(c_educ) + C(c_incm) + C(c_bd1_11) + C(c_bs3_1) + age_n"
res_m1 = fit_wls(f"bmi ~ {EXPO}", sub_df)
res_m2 = fit_wls(f"bmi ~ {EXPO} + {COV_TERMS}", sub_df)

# 3) 계수 이름을 정확히 하나만 찾는 함수
def term(res, level):
    hits = [n for n in res.params.index if n.startswith('C(grp') and n.endswith(f'[T.{level}]')]
    assert len(hits) == 1, hits
    return hits[0]

def fmt(res, level):
    t = term(res, level)
    b, se, p = res.params[t], res.bse[t], res.pvalues[t]
    lo, hi = res.conf_int().loc[t]
    ps = f"{p:.3f}" if p >= 0.001 else "< 0.001"
    return f"β = {b:.3f} (SE = {se:.3f}); 95% CI [{lo:.3f}, {hi:.3f}]; p = {ps}"

# 4) 검증: 모형 1 계수 = 완전사례 내 가중평균 차이여야 함
wm = {g: np.average(d['bmi'], weights=d['w']) for g, d in sub_df.groupby('grp', observed=True)}
print("완전사례 내 가중 평균 BMI:", {k: round(v, 3) for k, v in wm.items()})
print("가중평균 차이  high-low: %.3f / mid-low: %.3f" % (wm['high'] - wm['low'], wm['mid'] - wm['low']))
print("모형1 계수     high: %.3f / mid: %.3f" % (res_m1.params[term(res_m1, 'high')], res_m1.params[term(res_m1, 'mid')]))
assert np.isclose(res_m1.params[term(res_m1, 'high')], wm['high'] - wm['low'])
assert np.isclose(res_m1.params[term(res_m1, 'mid')],  wm['mid']  - wm['low'])
print("집단별 인원:", sub_df['grp'].value_counts().to_dict(), "| 전체 N =", n_complete)

# 5) Table 2 (화면용 HTML 표 + 복사용 마크다운)
from IPython.display import display, HTML

def fmt_html(res, level):
    t = term(res, level)
    b, se, p = res.params[t], res.bse[t], res.pvalues[t]
    lo, hi = res.conf_int().loc[t]
    ps = f"p = {p:.3f}" if p >= 0.001 else "p < 0.001"
    return f"β = {b:.3f} (SE = {se:.3f})<br>95% CI: [{lo:.3f}, {hi:.3f}]<br>{ps}"

table2 = pd.DataFrame([
    {"모형": "모형 1 (미보정)", "분석 인원 (N)": f"{n_complete:,}명",
     "저빈도 외식군 (주 1–2회 이하)": "Ref.",
     "중빈도 외식군 (주 3–4회)": fmt_html(res_m1, 'mid'),
     "고빈도 외식군 (주 5회 이상)": fmt_html(res_m1, 'high')},
    {"모형": "모형 2 (보정)", "분석 인원 (N)": f"{n_complete:,}명",
     "저빈도 외식군 (주 1–2회 이하)": "Ref.",
     "중빈도 외식군 (주 3–4회)": fmt_html(res_m2, 'mid'),
     "고빈도 외식군 (주 5회 이상)": fmt_html(res_m2, 'high')},
])

print(f"=== [Table 2] 외식 빈도에 따른 BMI 차이 (kg/m², 기준: 저빈도군) | Table 1 대비 {diff_count}명 제외 ===")
display(HTML(table2.to_html(index=False, escape=False)))

완전사례 내 가중 평균 BMI: {'low': np.float64(24.013), 'mid': np.float64(24.117), 'high': np.float64(24.675)}
가중평균 차이  high-low: 0.662 / mid-low: 0.104
모형1 계수     high: 0.662 / mid: 0.104
집단별 인원: {'low': 2961, 'high': 1860, 'mid': 781} | 전체 N = 5602
=== [Table 2] 외식 빈도에 따른 BMI 차이 (kg/m², 기준: 저빈도군) | Table 1 대비 163명 제외 ===


모형,분석 인원 (N),저빈도 외식군 (주 1–2회 이하),중빈도 외식군 (주 3–4회),고빈도 외식군 (주 5회 이상)
모형 1 (미보정),"5,602명",Ref.,"β = 0.104 (SE = 0.198)95% CI: [-0.284, 0.493]p = 0.599","β = 0.662 (SE = 0.141)95% CI: [0.386, 0.938]p < 0.001"
모형 2 (보정),"5,602명",Ref.,"β = 0.194 (SE = 0.197)95% CI: [-0.193, 0.581]p = 0.326","β = 0.474 (SE = 0.153)95% CI: [0.175, 0.774]p = 0.002"


#각주

1. 설계 및 분석 모형:
국민건강영양조사 가중치(wt_tot)와 집락변수(psu)를 반영한 가중 선형회귀모형(WLS)을 적합하였고, 표준오차는 psu 단위의 군집 강건 표준오차를 사용하였다. 층화변수(kstrata)는 표준오차 계산에 직접 반영하지 않은 근사 추정이다.

2. 분석 대상 처리:
분석 대상(19세 이상, 외식 빈도·BMI·설계변수 유효)에서 보정변수 결측자를 제외한 완전 사례 서브셋으로 분석하였다. 모형 1과 모형 2는 같은 대상자를 사용하였다.

3. 결측 처리 방법:
노출(외식 빈도), 결과(BMI) 및 필수 보정변수(성별, 연령, 거주 지역, 교육수준, 소득 4분위, 음주 빈도, 현재 흡연 여부) 중 하나라도 결측(모름·무응답 포함)이면 제외하였다.

4. 모형 종류 및 보정변수 목록:
모형 1 (Unadjusted): 외식 빈도 범주 단독 투입 (미보정 단변량 모형) /
모형 2 (Adjusted): 모형 1에 성별, 연령, 거주 지역, 교육수준, 소득 4분위, 음주 빈도, 현재 흡연 여부를 보정한 다변량 모형

5. 분석 인원:
Table 1의 분석 대상은 5,765명이며, 보정변수 결측 163명이 제외된 5,602명이 모형 1과 2의 분석 인원이다.

6. 결과 해석 단위 및 자료:
계수(β)는 저빈도 외식군(주 1-2회 이하) 대비 BMI 평균 차이(kg/m)이다. / 질병관리청 국민건강영양조사 제9기 2024년.